# Heritage short: original score with ACE-Step 1.5

Makes **3 instrumental candidates** (fixed seeds) for *Youth of the Future*, then zips them with a `manifest.json`
(model, commit, GPU, seed, seconds) so the local pipeline can log them in `runs.jsonl`.

**Before running:** right panel → *Session options* → **Accelerator: GPU T4 x2**, **Internet: On**.
Then **Run All**. Takes about 15–25 min (most of it installing and downloading the model once).

When it finishes, download `score_candidates.zip` from the *Output* panel (right side, `/kaggle/working`).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# install ACE-Step 1.5 the documented way (git clone + uv sync)
!pip install -q uv
!git clone --depth 1 https://github.com/ACE-Step/ACE-Step-1.5.git /kaggle/working/ACE-Step-1.5
%cd /kaggle/working/ACE-Step-1.5
!uv sync 2>&1 | tail -n 5

In [ ]:
# download the model weights once (also happens automatically on first use)
!uv run acestep-download 2>&1 | tail -n 5 || echo "download command not available; weights will download on first use"

In [ ]:
%%writefile /kaggle/working/gen.py
# Runs inside the ACE-Step-1.5 uv environment: `uv run python /kaggle/working/gen.py`
import json, os, shutil, subprocess, time

import torch
from acestep.handler import AceStepHandler
from acestep.llm_inference import LLMHandler
from acestep.inference import GenerationParams, GenerationConfig, generate_music

# --- the brief for the score (same caption as storyboard.json -> music.caption) ---
CAPTION = ("gentle nostalgic instrumental, soft oud and warm synth pads, light hand percussion, "
           "a simple playful music-box motif, slow, no vocals, no strong lead melody")
SEEDS = [11, 22, 33]        # three candidates; fixed seeds so any one can be re-made exactly
DURATION = 85               # the film is ~79 s; the pipeline trims, loops and fades to length
BPM, KEY = 80, "A minor"
STEPS = 8                   # turbo model: 8 steps recommended by the docs
OUT = "/kaggle/working/candidates"

os.makedirs(OUT, exist_ok=True)
root = os.getcwd()
commit = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("GPU:", torch.cuda.get_device_name(0), "| ACE-Step commit:", commit)

t0 = time.time()
dit = AceStepHandler()
dit.initialize_service(project_root=root, config_path="acestep-v15-turbo", device="cuda")
llm = LLMHandler()
try:
    # the 5Hz LM only plans; we switch planning off (thinking=False), but the API expects a handler
    llm.initialize(checkpoint_dir=os.path.join(root, "checkpoints"), lm_model_path="acestep-5Hz-lm-0.6B", backend="pt", device="cuda")
except Exception as exc:
    print("LM init failed (continuing without planning):", exc)
load_s = time.time() - t0
print(f"models loaded in {load_s:.0f}s")

rows = []
for seed in SEEDS:
    params = GenerationParams(task_type="text2music", caption=CAPTION, lyrics="[Instrumental]", instrumental=True,
                              duration=DURATION, bpm=BPM, keyscale=KEY, inference_steps=STEPS, shift=3.0,
                              seed=seed, thinking=False)
    config = GenerationConfig(batch_size=1, use_random_seed=False, seeds=[seed], audio_format="wav")
    t = time.time()
    try:
        result = generate_music(dit, llm, params, config, save_dir=OUT)
        ok, err = result.success, result.error
    except Exception as exc:
        ok, err, result = False, repr(exc), None
    secs = round(time.time() - t, 1)
    if not ok:
        print(f"seed {seed}: FAILED after {secs}s: {err}")
        rows.append({"seed": seed, "status": f"error: {err}"[:300], "seconds": secs})
        continue
    audio = result.audios[0]
    name = f"seed{seed}.wav"
    shutil.move(audio["path"], os.path.join(OUT, name))
    rows.append({"file": name, "seed": seed, "status": "ok", "seconds": secs, "sample_rate": audio.get("sample_rate")})
    print(f"seed {seed}: ok in {secs}s -> {name}")

manifest = {"model": "ACE-Step 1.5 (acestep-v15-turbo)", "repo_commit": commit, "gpu": torch.cuda.get_device_name(0),
            "caption": CAPTION, "bpm": BPM, "key": KEY, "duration": DURATION, "inference_steps": STEPS,
            "load_seconds": round(load_s, 1), "candidates": rows}
json.dump(manifest, open(os.path.join(OUT, "manifest.json"), "w"), indent=2)
print(json.dumps(manifest, indent=2))


In [ ]:
%cd /kaggle/working/ACE-Step-1.5
# Kaggle sets MPLBACKEND to its inline plot backend, which the uv environment's matplotlib rejects (FAILURES.md F8)
!MPLBACKEND=Agg uv run python /kaggle/working/gen.py

In [ ]:
# package the candidates for download
!cd /kaggle/working && rm -f score_candidates.zip && zip -r -q score_candidates.zip candidates && ls -lh score_candidates.zip candidates

## Listen and choose
Play the three files in `candidates/` (click them in the Output panel). Note for each: does it sit **under** a voice
(no busy lead melody)? Does it feel warm and nostalgic? Then download the zip and, in the repo:

```
uv run python -m pipeline music import path\to\score_candidates.zip
uv run python -m pipeline music choose seed22      # the one you picked
uv run python -m pipeline film
```

**If it fails:** copy the error into FAILURES.md. A common one on T4 is bfloat16 not being supported; the fallback is the
pipeline's placeholder drone, which the brief allows as long as it is logged honestly.